# Controlled Google Colab Training

This notebook is a thin human-operated entrypoint for CON-011. It runs one full experiment from an exact Git commit and persists artifacts to Google Drive. Notebook execution is not experiment completion until the returned bundle passes local validation.

In [ ]:
# Human-supplied immutable run parameters.
REPOSITORY_URL = "https://github.com/dkumar-23/RL_Lunar-Lander"
GIT_COMMIT_SHA = ""
EXPERIMENT_ID = ""  # EXP-001 through EXP-004
RUN_ID = ""  # RUN-NNN
CONFIGURATION_PATH = ""
RANDOM_SEED = 0
DRIVE_ARTIFACT_ROOT = "/content/drive/MyDrive/RL_Lunar-Lander/runs"
MINIMUM_FREE_DRIVE_BYTES = 5_000_000_000

In [ ]:
import re
import subprocess
import sys
from pathlib import Path

from google.colab import drive

if re.fullmatch(r"(?:[0-9a-f]{40}|[0-9a-f]{64})", GIT_COMMIT_SHA) is None:
    raise ValueError("GIT_COMMIT_SHA must be a complete immutable commit SHA")
if re.fullmatch(r"EXP-00[1-4]", EXPERIMENT_ID) is None:
    raise ValueError("EXPERIMENT_ID must be EXP-001 through EXP-004")
if re.fullmatch(r"RUN-[0-9]{3}", RUN_ID) is None:
    raise ValueError("RUN_ID must match RUN-NNN")
if not CONFIGURATION_PATH:
    raise ValueError("CONFIGURATION_PATH is required")
if MINIMUM_FREE_DRIVE_BYTES <= 0:
    raise ValueError("MINIMUM_FREE_DRIVE_BYTES must be positive")

drive.mount("/content/drive")
drive_root = Path(DRIVE_ARTIFACT_ROOT).resolve()
if Path("/content/drive") not in (drive_root, *drive_root.parents):
    raise ValueError("DRIVE_ARTIFACT_ROOT must be under /content/drive")
drive_root.mkdir(parents=True, exist_ok=True)

In [ ]:
workspace = Path("/content/work/RL_Lunar-Lander")
if workspace.exists():
    raise RuntimeError(f"Refusing to reuse existing workspace: {workspace}")
workspace.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(["git", "init", str(workspace)], check=True)
subprocess.run(["git", "-C", str(workspace), "remote", "add", "origin", REPOSITORY_URL], check=True)
subprocess.run(["git", "-C", str(workspace), "fetch", "--depth=1", "origin", GIT_COMMIT_SHA], check=True)
subprocess.run(["git", "-C", str(workspace), "checkout", "--detach", "FETCH_HEAD"], check=True)
resolved_commit = subprocess.run(["git", "-C", str(workspace), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
worktree_status = subprocess.run(["git", "-C", str(workspace), "status", "--porcelain"], check=True, capture_output=True, text=True).stdout.strip()
if resolved_commit != GIT_COMMIT_SHA:
    raise RuntimeError(f"Resolved commit {resolved_commit} differs from requested commit")
if worktree_status:
    raise RuntimeError("Training requires a clean detached worktree")

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(workspace / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
    raise RuntimeError("A visible CUDA GPU is required for canonical training")
print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
run_root = drive_root / GIT_COMMIT_SHA / EXPERIMENT_ID / RUN_ID
if run_root.exists():
    raise RuntimeError(f"Refusing to overwrite an existing run: {run_root}")
command = [
    sys.executable,
    "-m", "scripts.train",
    "--execution-context", "colab-full",
    "--repository-root", str(workspace),
    "--expected-commit", GIT_COMMIT_SHA,
    "--experiment-id", EXPERIMENT_ID,
    "--run-id", RUN_ID,
    "--config", str(workspace / CONFIGURATION_PATH),
    "--seed", str(RANDOM_SEED),
    "--output", str(run_root),
    "--minimum-free-drive-bytes", str(MINIMUM_FREE_DRIVE_BYTES),
]
subprocess.run(command, check=True, cwd=workspace)